# DAPPC LAB 4 - Construction of a clinical score to monitor clinical status evolution

This notebook guides you in the implementation of a **fuzzy inference system (FIS)** to construct a clinical score from the subset of features selected during LAB3.

## Recommended workflow
1. Load the dataset and define the outcome classes.
2. Select the subset of features identified in LAB3.
3. Implement the membership functions already defined graphically.
4. Perform hierarchical clustering separately within each outcome class.
5. Cut the dendrograms, identify the clusters, and compute the cluster centroids.
6. Convert each centroid into a fuzzy rule.
7. Build the fuzzy inference system and compute the final clinical risk score.
8. Analyze the score distribution across the three clinical outcomes.


## 0. Setup

You may have to install some dependencies:
1. the scikit-fuzzy package -> pip install -U scikit-fuzzy
2. the networkx package -> pip install networkx


In [8]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import MinMaxScaler

import skfuzzy as fuzz
from skfuzzy import control as ctrl

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## 1. Load data

Load the dataset obtained after LAB3.

The dataset must contain all the subjects but only some columns:
- the identifiers (which must **not** be used in the model),
- the selected input features,
- the outcome column.


In [ ]:
file_path = ...  
sheet_name = ...                                       

df = pd.read_excel(file_path, sheet_name=sheet_name)
print('Dataset shape:', df.shape)
display(df.head())


## 2. Define the outcome column and the selected feature subset

Use the subset of predictors identified in LAB3.

Do **not** include IDs or other non-informative columns.


In [ ]:
outcome_col = 'outcome' 
selected_features = []

print('Selected features:', selected_features)
print('Number of selected features:', len(selected_features))
print('Class distribution:')
display(df[outcome_col].value_counts(dropna=False))


___
# PART 1: MFs DEFINITION

## 3. Define the fuzzy variables and the membership functions

The membership functions were already identified graphically.
In this step, simply translate them into Python.

For each input feature (`Antecedent`):
1. define the range (i.e., the `universe`),
2. create the fuzzy variable,
3. assign the MFs using `trimf` (triangular MF) and/or `trapmf` (trapezoidal MF).

The final output (`Consequent`) is a continuous variable called `clinical_risk`.


In [ ]:
# ============================================================
# Example structure for the input variables
# Repeat the same logic for all selected features
# ============================================================

fuzzy_inputs = {}

# Example for one feature:
# age = ctrl.Antecedent(np.arange(18, 101, 1), 'age')
# age['young'] = fuzz.trapmf(age.universe, [18, 18, 30, 45])
# age['middle'] = fuzz.trimf(age.universe, [35, 50, 65])
# age['old'] = fuzz.trapmf(age.universe, [55, 70, 100, 100])
# fuzzy_inputs['age'] = age

...

# Output variable (you may call it "clinical_risk")
...


### 4. Check the membership functions

Visualize all the defined membership functions and compare them with the graphical design developed previously.

This step is important, before moving on, make sure that:
- the numerical ranges are correct,
- the linguistic terms (names of the MFs) are assigned correctly,
- the shapes of the membership functions match the ones defined graphically.

If needed, revise the parameters before proceeding.

In [ ]:
for feature_name, fuzzy_var in fuzzy_inputs.items():
    print(f"Membership functions for: {feature_name}")
    fuzzy_var.view()

clinical_risk.view()

___
# PART 2: RULES IDENTIFICATION

### 5. Prepare data for clustering
Before splitting the subjects by outcome, apply Min-Max normalization to the selected features.
This step ensures that all variables contribute comparably to the hierarchical clustering procedure.

Note: clustering is performed in the normalized feature space, while the fuzzy membership functions are defined in the original clinical domain. Therefore, cluster centroids must be transformed back to the original scale before generating the fuzzy rules.

In [ ]:
# Select the features identified in LAB3
X = df[selected_features].copy()

# Apply Min-Max normalization before splitting by outcome
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=selected_features,
    index=df.index
)

# Add the outcome column for the next steps
df_scaled = X_scaled.copy()
df_scaled[outcome_column] = df[outcome_column]

## 5. Split the subjects according to the outcome

We want to identify the rules using hierarchical clustering, performed **separately** for each outcome class.
This allows us to identify the main patterns within each clinical condition.


In [ ]:
...

## 6. Hierarchical clustering within each outcome class

For each class:
1. use only the selected features,
3. compute the linkage matrix (method complete, metric cityblock),
4. visualize the dendrogram,
5. cut the dendrogram.


In [ ]:
...


## 7. Cut the dendrograms and assign the cluster labels

Choose the cut strategy and assign one cluster label to each subject.

Two common options are:
- `criterion='maxclust'`: impose the number of clusters,
- `criterion='distance'`: impose a cut threshold.


In [ ]:
...


## 8. Compute the cluster centroids

The centroid of each cluster is used as a numerical prototype.
Each prototype will then be converted into one or more fuzzy rule(s).

Remember that the clustering was performed on normalized data but the MFs are in the original feature space.


In [ ]:
...

## 9. Identify the rules

From here you should work firstly on the MFs graphically designed to identify the rules: the antecedents are determined by the values of the features of the centroids, the consequent is determined by the class of the subjects you're working on.

Collect all the rules in a table (maybe and excel file can be useful) and then check if there're duplicate rules and clear it. 
The file must have all the features and the outcome as header columns and the values inside the cells must be the same "names" of the MFs.



## 10. Build the fuzzy rules

Now you have to implement on Python the rules identified. For simplicity, you can load the excel file obtained above and use it io import the rules. 

In [ ]:
# Load the file with the rules
rules_df = pd.read_excel(...)
input_columns = [...]

rules = []

for _, row in rules_df.iterrows():
    antecedent = fuzzy_inputs[input_columns[0]][row[input_columns[0]]]
    
    for col in input_columns[1:]:
        antecedent = antecedent & fuzzy_inputs[col][row[col]]
    
    rule = ctrl.Rule(antecedent, clinical_risk[row[...]]) # name of the consequent
    rules.append(rule)

    

___
# PART 3:  BUILD AND USE THE FIS

## 11. Build the fuzzy inference system

Combine all the rules into a single fuzzy inference system (FIS) and test it on one example patient.


In [ ]:
# Build the FIS
risk_ctrl = ctrl.ControlSystem(rules)

# Test on one patient (extract one patient from the dataset and ensure that the system works)
example_patient = ...
risk_sim = ctrl.ControlSystemSimulation(risk_ctrl)

for feature in selected_features:
    risk_sim.input[feature] = example_patient[feature]

risk_sim.compute()

print("Available outputs:", risk_sim.output)

# "clinical_risk" is the name of the Consequent of the FIS (if you named it differently, change it accordingly to make it works)
if "clinical_risk" in risk_sim.output:
    print("Example patient index:", example_patient.name)
    print("Observed outcome:", example_patient[outcome_col])
    print("Predicted clinical risk score:", risk_sim.output["clinical_risk"])
else:
    print("No risk output was produced. Probably no rule was activated.")

## 12. Use the FIS to compute the clinical risk for all the patients of the analyzed cluster

Apply the fuzzy inference system to all the subjects in the working cluster. 
Use the code at step 11 and generalize it for the entire cluster.


In [ ]:
...

## 13. Results analysis

Now that you have the FIS and you applied it to the subjects of the cluster you're working on, youshould analyze and discuss the results obtained. Some suggestions:
1. Analyze the percentage (or the number) of patients without a clinical score computed compared to the others;
2. Visualize the distribution of the clinical risk score obtained over the three classes of the outcome;
3. Convert the clinical score to the classification (0/1/2) using the MFs you previously designed (maybe you can use the confusion matrix).
4. Discuss if there're some changes over the past steps (i.e., MFs definition and rules identification) to improve the results (e.g., how to reduce the number of unclassified subjects...)
